In [19]:
import pandas as pd
import numpy as np
import altair as alt

import warnings
warnings.filterwarnings('ignore')

In [5]:
# Read in all data

nominees = pd.read_csv("Nominees.csv")
winners_bo = pd.read_csv("Winners and BO.csv")
total_bo = pd.read_csv("Total BO.csv")

In [21]:
# Dashboard

# Allow users to select points by year on all graphs
selection = alt.selection(type = "multi", fields = ["Year"])

# Bar graph of total domestic b.o. of nominees by year
bar = alt.Chart(total_bo, title = "Total Domestic Box Office of All Best Picture Nominees by Year").mark_bar(size = 20).encode(
    x = alt.X("Year", axis = alt.Axis(format = "d"), scale = alt.Scale(domain = [2009.5, 2026.5])),
    y = alt.Y("Total Domestic BO", title = "Inflation-Adjusted Box Office", scale = alt.Scale(domain = [0, 2800000000])),
    color = alt.condition(selection, alt.value("steelblue"), alt.value("lightsteelblue")),
    tooltip = [alt.Tooltip("Total Domestic BO", title = "Total Domestic Box Office of All Nominees", format = "$,.0f"),
             "Highest Grossing Film", 
             alt.Tooltip("Highest Grossing Film BO", title = "Highest Grossing Film's Box Office", format = "$,.0f")]
).properties(
    width = 600,
    height = 200
).add_selection(selection)



# Scatterplot of Oscar ratings relative to total domestic b.o.
scatter = alt.Chart(total_bo, title = "Oscar Viewers vs. Total Domestic Box Office of Nominees").mark_circle(size = 150).encode(
    x = alt.X("Total Domestic BO", title = "Inflation-Adjusted Box Office"),
    y = "Oscar Viewers (in millions)",
    color = alt.condition(selection, alt.value("steelblue"), alt.value("lightgrey")),
    tooltip = ["Year", "Oscar Viewers (in millions)",
              alt.Tooltip("Total Domestic BO", title = "Total Domestic Box Office of All Nominees", format = "$,.0f")]
).add_selection(selection)



# Filter csv of winners to only those from 2010-2026
recent_winners = winners_bo[winners_bo["Year"] > 2009]

# Line graph of Oscar ratings by year
line = alt.Chart(recent_winners, title = "Oscar Ratings by Year").mark_line(point = True).encode(
    x = alt.X("Year", axis = alt.Axis(format = "d"), scale = alt.Scale(domain = [2009.5, 2026.5])),
    y = "Oscar Viewers (in millions)",
    color = alt.condition(selection, alt.value("steelblue"), alt.value("lightgrey")),
    tooltip = ["Year", "Oscar Viewers (in millions)"]
).properties(
    width = 600,
    height = 200
).add_selection(selection)



# Combine graphs in dashboard, bar and line on top of each other and scatter to right
dashboard = bar & line | scatter
dashboard

alt.HConcatChart(...)

In [23]:
# Swarm Plot

# Add jitter to values in swarm plot
nominees["x_jitter"] = nominees["Year"] + np.random.uniform(-0.25, 0.25, len(nominees)) 


# Swarm plot of box office values for of all nominees in a given year
# Tell people to scroll/drag to zoom in and out explore the plot, double click to reset
swarm = alt.Chart(nominees, title = "Box Office of Best Picture Nominees By Year").mark_circle(size = 200).encode(
    x = alt.X("x_jitter", 
              title = "Year",
              axis = alt.Axis(format = "d"), 
              scale = alt.Scale(domain = [2009.5, 2026.5])),
    y = alt.Y("Inflation-Adjusted Domestic Box Office", 
              scale = alt.Scale(domain = [0, 1250000000]), 
              axis = alt.Axis(tickCount = 10)),
    color = alt.Color("Year", 
                      scale = alt.Scale(scheme = "rainbow")).legend(None),
    tooltip = ["Film", 
               "Year", 
               alt.Tooltip("Inflation-Adjusted Domestic Box Office", title="Box Office", format="$,.0f")
              ]
).properties(
    width=1000,
    height=500
).interactive()

swarm

alt.Chart(...)

In [ ]:
# Exports both charts as JSON, so they both display on the same webpage, and there isn't redundant code

two_charts_template = """
<!DOCTYPE html>
<html>
<head>
  <script src="https://cdn.jsdelivr.net/npm/vega@{vega_version}"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@{vegalite_version}"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@{vegaembed_version}"></script>
</head>
<body>

<div id="vis1"></div>
<div id="vis2"></div>

<script type="text/javascript">
  vegaEmbed('#vis1', {spec1}).catch(console.error);
  vegaEmbed('#vis2', {spec2}).catch(console.error);
</script>
</body>
</html>
"""

with open('charts.html', 'w') as f:
    f.write(two_charts_template.format(
        vega_version = alt.VEGA_VERSION,
        vegalite_version = alt.VEGALITE_VERSION,
        vegaembed_version = alt.VEGAEMBED_VERSION,
        spec1 = swarm.to_json(indent=None),
        spec2 = dashboard.to_json(indent=None),))